# 📊 Notebook 03 — A gramática da culpa: o que o BR busca no Google

**Projeto:** Déficit de Dado — Edição 001  
**Artigo:** Treinar pra poder comer: a relação das mulheres BR com exercício e culpa  
**Autor:** Lanadeilana  

Este notebook coleta dados reais do Google Trends (via pytrends) para analisar  
como as buscas brasileiras sobre fitness, dieta e culpa evoluíram entre 2019–2024.

**Hipótese:** Se a cultura da culpa fitness é estrutural, termos como  
'compensar comida', 'treino para emagrecer', 'como queimar calorias'  
devem ter crescido junto com o mercado fitness.

**Ferramenta:** pytrends (wrapper não-oficial da API do Google Trends)  
**Documentação:** https://github.com/GeneralMills/pytrends

> ℹ️ O Google Trends retorna índices relativos (0-100), não volumes absolutos.  
> 100 = pico de interesse no período. Os dados são anonimizados e agregados pelo Google.

In [ ]:
# Instalação
# !pip install pytrends pandas matplotlib seaborn

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from pytrends.request import TrendReq

CORES = {
    'principal': '#7F77DD',
    'secundaria': '#1D9E75',
    'destaque': '#D85A30',
    'rosa': '#D4537E',
    'amarelo': '#EF9F27',
    'neutro': '#888780',
    'fundo': '#F8F7F2'
}

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150
})

# Inicializa conexão com Google Trends (pt-BR, fuso de Brasília = UTC-3 = 180min)
pytrends = TrendReq(hl='pt-BR', tz=180, timeout=(10, 25), retries=3, backoff_factor=0.5)

print('pytrends conectado. Pronto para coletar.')

## 1. Termos de culpa fitness

Vamos comparar termos que revelam a relação entre exercício e compensação alimentar.  

> ⏱️ Pytrends tem rate limit do Google. Se receber erro 429, aguarde ~60s e tente novamente.

In [ ]:
# Grupo 1: termos de culpa e compensação
# O pytrends aceita no máximo 5 termos por chamada

termos_culpa = [
    'compensar comida',
    'treino para emagrecer',
    'como queimar calorias',
    'treinar pra comer',
    'culpa depois de comer'
]

pytrends.build_payload(
    kw_list=termos_culpa,
    cat=0,
    timeframe='2019-01-01 2024-12-31',
    geo='BR'
)

df_culpa = pytrends.interest_over_time()

if not df_culpa.empty:
    df_culpa = df_culpa.drop(columns=['isPartial'], errors='ignore')
    print(f'Dados coletados: {len(df_culpa)} semanas')
    print(df_culpa.describe().round(1))
else:
    print('Sem dados — tente novamente em 60 segundos (rate limit)')

In [ ]:
# Grupo 2: termos de dieta e controle
time.sleep(5)  # pequena pausa entre requisições

termos_dieta = [
    'detox',
    'cutting academia',
    'dieta restritiva',
    'transtorno alimentar',
    'compulsao alimentar'
]

pytrends.build_payload(
    kw_list=termos_dieta,
    cat=0,
    timeframe='2019-01-01 2024-12-31',
    geo='BR'
)

df_dieta = pytrends.interest_over_time()

if not df_dieta.empty:
    df_dieta = df_dieta.drop(columns=['isPartial'], errors='ignore')
    print(f'Dados coletados: {len(df_dieta)} semanas')
    print(df_dieta.describe().round(1))
else:
    print('Sem dados — tente novamente em 60 segundos')

## 2. Visualização — a linha do tempo da culpa

In [ ]:
# Visualização dos termos de culpa ao longo do tempo
# (rode após coletar os dados nas células acima)

fig, axes = plt.subplots(2, 1, figsize=(13, 10))
fig.patch.set_facecolor(CORES['fundo'])

cores_linhas = [CORES['principal'], CORES['rosa'], CORES['destaque'],
                CORES['secundaria'], CORES['amarelo']]

# --- Painel 1: termos de culpa e compensação ---
ax1 = axes[0]
ax1.set_facecolor(CORES['fundo'])

if not df_culpa.empty:
    # Suaviza com média móvel de 4 semanas
    df_culpa_smooth = df_culpa.rolling(4, center=True).mean()
    
    for col, cor in zip(df_culpa_smooth.columns, cores_linhas):
        ax1.plot(df_culpa_smooth.index, df_culpa_smooth[col],
                 color=cor, linewidth=1.8, label=col, alpha=0.9)
    
    # Marca pandemia
    ax1.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-12-01'),
               alpha=0.08, color=CORES['neutro'])
    ax1.text(pd.Timestamp('2020-06-01'), ax1.get_ylim()[1] * 0.9,
             'Pandemia', ha='center', fontsize=9, color=CORES['neutro'])
else:
    ax1.text(0.5, 0.5, 'Execute a coleta de dados primeiro',
             ha='center', va='center', transform=ax1.transAxes,
             fontsize=12, color=CORES['neutro'])

ax1.set_title('Buscas sobre compensação alimentar no Brasil (2019–2024)',
              fontsize=12, loc='left', pad=10)
ax1.set_ylabel('Interesse relativo (0–100)')
ax1.legend(loc='upper left', fontsize=9)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# --- Painel 2: termos de dieta e transtornos ---
ax2 = axes[1]
ax2.set_facecolor(CORES['fundo'])

if not df_dieta.empty:
    df_dieta_smooth = df_dieta.rolling(4, center=True).mean()
    
    for col, cor in zip(df_dieta_smooth.columns, cores_linhas):
        ax2.plot(df_dieta_smooth.index, df_dieta_smooth[col],
                 color=cor, linewidth=1.8, label=col, alpha=0.9)
    
    ax2.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-12-01'),
               alpha=0.08, color=CORES['neutro'])
else:
    ax2.text(0.5, 0.5, 'Execute a coleta de dados primeiro',
             ha='center', va='center', transform=ax2.transAxes,
             fontsize=12, color=CORES['neutro'])

ax2.set_title('Buscas sobre dieta, transtornos e controle alimentar no Brasil (2019–2024)',
              fontsize=12, loc='left', pad=10)
ax2.set_ylabel('Interesse relativo (0–100)')
ax2.set_xlabel('Ano')
ax2.legend(loc='upper left', fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.text(0.08, -0.02,
         'Fonte: Google Trends via pytrends | geo=BR | 2019-2024 | índice relativo (pico = 100)',
         fontsize=8, color=CORES['neutro'])

plt.suptitle('Déficit de Dado • Edição 001', y=1.01, fontsize=10,
             color=CORES['neutro'], ha='left', x=0.08)
plt.tight_layout()
plt.savefig('fig06_trends_culpa_fitness.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura salva: fig06_trends_culpa_fitness.png')

## 3. Sazonalidade — quando a culpa tem mais audiência?

In [ ]:
# Análise de sazonalidade mensal
# Hipótese: picos em janeiro (resolução de ano novo) e pré-carnaval

if not df_culpa.empty:
    df_culpa['mes'] = df_culpa.index.month
    df_culpa['mes_nome'] = df_culpa.index.strftime('%b')
    
    # Média por mês (todos os anos)
    sazonalidade = df_culpa.groupby('mes').mean().round(1)
    
    # Seleciona os 2 termos mais relevantes pra sazonalidade
    termos_viz = ['treino para emagrecer', 'compensar comida']
    termos_viz = [t for t in termos_viz if t in sazonalidade.columns]
    
    if termos_viz:
        fig, ax = plt.subplots(figsize=(12, 5))
        fig.patch.set_facecolor(CORES['fundo'])
        ax.set_facecolor(CORES['fundo'])
        
        meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
                 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
        
        for termo, cor in zip(termos_viz, [CORES['principal'], CORES['rosa']]):
            if termo in sazonalidade.columns:
                vals = sazonalidade[termo].values
                ax.plot(range(12), vals, color=cor, linewidth=2.5,
                        marker='o', markersize=7, label=termo)
                ax.fill_between(range(12), vals, alpha=0.08, color=cor)
        
        # Marca picos esperados
        ax.axvspan(-0.5, 1.5, alpha=0.05, color=CORES['destaque'])
        ax.text(0.5, ax.get_ylim()[1] * 0.95, 'Virada\nde ano',
                ha='center', fontsize=8, color=CORES['destaque'])
        
        ax.set_xticks(range(12))
        ax.set_xticklabels(meses)
        ax.set_ylabel('Interesse médio relativo')
        ax.set_title('Sazonalidade: quando o Brasil mais busca por compensação fitness?',
                     fontsize=12, loc='left', pad=10)
        ax.legend(fontsize=10)
        
        plt.tight_layout()
        plt.savefig('fig07_sazonalidade.png', bbox_inches='tight', dpi=150)
        plt.show()
        print('Figura salva: fig07_sazonalidade.png')
else:
    print('Sem dados para sazonalidade — execute a coleta primeiro.')

## 4. Exportar dados coletados

In [ ]:
# Salva os dados brutos para reprodutibilidade
if not df_culpa.empty:
    df_culpa.to_csv('dados_trends_culpa_fitness.csv', encoding='utf-8-sig')
    print('Dados exportados: dados_trends_culpa_fitness.csv')

if not df_dieta.empty:
    df_dieta.to_csv('dados_trends_dieta_transtornos.csv', encoding='utf-8-sig')
    print('Dados exportados: dados_trends_dieta_transtornos.csv')

print()
print('=== PARA REPRODUZIR ESTA ANÁLISE ===')
print('1. pip install pytrends pandas matplotlib')
print('2. Execute as células em ordem')
print('3. Se receber erro 429 (rate limit), aguarde 60s entre chamadas')
print('4. Os dados do Google Trends são relativos e podem variar levemente entre coletas')
print()
print('Artigo completo: https://substack.com/@lanadeilana')
print('Contato: LinkedIn /lanadeilana')